# PRISM Climate Download — Iowa EPA Stations

Downloads daily 4 km PRISM gridded climate data from Oregon State University
and extracts values at each EPA monitoring station location.

Only the dates that appear in the water quality dataset are downloaded,
keeping the total request count manageable. Rasters are never written to
disk — each ZIP is streamed, sampled in memory, then discarded.

**Variables downloaded**
| Code | Description | Unit |
|---|---|---|
| `tmax` | Daily maximum temperature | °C |
| `tmin` | Daily minimum temperature | °C |
| `ppt` | Daily total precipitation | mm |
| `tdmean` | Daily mean dew-point temperature | °C |

**Output**
- `data/tabular/climate/raw/prism-iowa-climate.csv`
  One row per station × date with one column per variable.

**Resuming**  
The notebook checks for an existing output file at startup. If found,
already-completed dates are skipped so the download can be interrupted
and restarted without re-fetching data.

In [2]:
import io
import time
import zipfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import requests
import rasterio
from pathlib import Path

VARIABLES  = ['tmax', 'tmin', 'ppt', 'tdmean']
PRISM_URL  = 'https://services.nacse.org/prism/data/get/us/4km/{var}/{date}'
OUT_FILE   = Path('../../data/tabular/climate/raw/prism-iowa-climate.csv')
PAUSE_SEC  = 1.0   # polite delay between requests
SAVE_EVERY = 50    # write CSV every N dates

## 1. Build inputs: station locations and observation dates

In [3]:
stations = pd.read_csv('../../data/tabular/water-quality/clean/epa-stations-clean.csv',
                       usecols=['MonitoringLocationIdentifier',
                                'LatitudeMeasure', 'LongitudeMeasure'])
stations = stations.dropna(subset=['LatitudeMeasure', 'LongitudeMeasure']).drop_duplicates()
coords = list(zip(stations['LongitudeMeasure'], stations['LatitudeMeasure']))
station_ids = stations['MonitoringLocationIdentifier'].tolist()
print(f'Stations: {len(stations)}')

wq = pd.read_csv('../../data/tabular/water-quality/clean/epa-wq-clean.csv',
                 usecols=['ActivityStartDateTime'])
obs_dates = sorted(
    pd.to_datetime(wq['ActivityStartDateTime']).dt.strftime('%Y%m%d').unique()
)
print(f'Unique observation dates: {len(obs_dates)}')
print(f'Range: {obs_dates[0]} – {obs_dates[-1]}')

Stations: 1666
Unique observation dates: 2464
Range: 20150105 – 20251225


## 2. Resume: skip dates already in the output file

In [4]:
if OUT_FILE.exists():
    existing = pd.read_csv(OUT_FILE, usecols=['date'])
    done_dates = set(existing['date'].astype(str).str.replace('-', ''))
    remaining = [d for d in obs_dates if d not in done_dates]
    print(f'Resuming: {len(done_dates)} dates done, {len(remaining)} remaining')
else:
    remaining = obs_dates
    print(f'Starting fresh: {len(remaining)} dates to download')

Starting fresh: 2464 dates to download


## 3. Download and sample

## Warning: 3-hour download ahead!

In [5]:
def fetch_prism_values(var, date_str, coords):
    """Download one PRISM daily ZIP and sample at all station coordinates.
    Returns array of shape (n_stations,), or None on failure."""
    url = PRISM_URL.format(var=var, date=date_str)
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
    except Exception as e:
        print(f'  WARN {var} {date_str}: {e}')
        return None

    # PRISM returns a plain-text error (not HTTP error) when rate-limited
    content_type = resp.headers.get('Content-Type', '')
    if 'text' in content_type or not zipfile.is_zipfile(io.BytesIO(resp.content)):
        print(f'  WARN {var} {date_str}: unexpected response — {resp.text[:120]}')
        return None

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        tif_name = next(
            (n for n in z.namelist() if n.endswith('.tif') and not n.endswith('.aux.xml')),
            None
        )
        if tif_name is None:
            print(f'  WARN {var} {date_str}: no .tif in ZIP')
            return None
        tif_bytes = z.read(tif_name)

    with rasterio.MemoryFile(tif_bytes) as mem:
        with mem.open() as src:
            nodata = src.nodata
            vals = np.array([v[0] for v in src.sample(coords)])
            if nodata is not None:
                vals = np.where(vals == nodata, np.nan, vals)
    return vals


records   = []
n_total   = len(remaining)
n_saved   = 0

for i, date_str in enumerate(remaining):
    date_fmt = f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}'
    row_vals = {sid: {'date': date_fmt, 'station_id': sid} for sid in station_ids}

    for var in VARIABLES:
        vals = fetch_prism_values(var, date_str, coords)
        time.sleep(PAUSE_SEC)
        for sid, v in zip(station_ids, (vals if vals is not None else [np.nan]*len(station_ids))):
            row_vals[sid][var] = v

    records.extend(row_vals.values())

    if (i + 1) % SAVE_EVERY == 0 or (i + 1) == n_total:
        chunk = pd.DataFrame(records)
        write_header = not OUT_FILE.exists()
        chunk.to_csv(OUT_FILE, mode='a', index=False, header=write_header)
        n_saved += len(records)
        records = []
        pct = 100 * (i + 1) / n_total
        print(f'  [{i+1}/{n_total}] {pct:.1f}%  saved {n_saved:,} rows so far')

print(f'Done. Total rows saved: {n_saved:,}')

  [50/2464] 2.0%  saved 83,300 rows so far
  [100/2464] 4.1%  saved 166,600 rows so far
  [150/2464] 6.1%  saved 249,900 rows so far
  [200/2464] 8.1%  saved 333,200 rows so far
  [250/2464] 10.1%  saved 416,500 rows so far
  [300/2464] 12.2%  saved 499,800 rows so far
  [350/2464] 14.2%  saved 583,100 rows so far
  [400/2464] 16.2%  saved 666,400 rows so far
  [450/2464] 18.3%  saved 749,700 rows so far
  [500/2464] 20.3%  saved 833,000 rows so far
  [550/2464] 22.3%  saved 916,300 rows so far
  WARN tmax 20170611: HTTPSConnectionPool(host='services.nacse.org', port=443): Read timed out. (read timeout=60)
  WARN tdmean 20170622: HTTPSConnectionPool(host='services.nacse.org', port=443): Read timed out.
  [600/2464] 24.4%  saved 999,600 rows so far
  [650/2464] 26.4%  saved 1,082,900 rows so far
  [700/2464] 28.4%  saved 1,166,200 rows so far
  [750/2464] 30.4%  saved 1,249,500 rows so far
  [800/2464] 32.5%  saved 1,332,800 rows so far
  [850/2464] 34.5%  saved 1,416,100 rows so far
  

## 4. Quick summary

In [6]:
out = pd.read_csv(OUT_FILE)
print('Output shape:    ', out.shape)
print('Date range:      ', out['date'].min(), '–', out['date'].max())
print('Stations:        ', out['station_id'].nunique())
print('Missing tmax:    ', out['tmax'].isna().sum())
print('Missing ppt:     ', out['ppt'].isna().sum())
out.head()

Output shape:     (4105024, 6)
Date range:       2015-01-05 – 2025-12-25
Stations:         1666
Missing tmax:     3332
Missing ppt:      1666


,date,station_id,tmax,tmin,ppt,tdmean
0,2015-01-05,USGS-05387490,-12.532,-23.183,0.0,-23.124
1,2015-01-05,USGS-05411260,-11.283,-22.664,0.0,-21.472
2,2015-01-05,USGS-05412400,-11.180,-22.892,0.0,-21.503
3,2015-01-05,USGS-05412500,-10.361,-22.082,0.0,-21.032
4,2015-01-05,USGS-05416900,-11.544,-22.294,0.0,-21.201
